In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# -----------------------------
# 1️⃣ IMPORTS
# -----------------------------
import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# -----------------------------
# 2️⃣ CONFIG
# -----------------------------
# Path to dataset on your Google Drive
DATA_ROOT = "/content/drive/MyDrive/Minor-project-ML-training-data-images/Phase1_224x224_RGB"

BATCH_SIZE = 32
NUM_EPOCHS = 15
LR = 1e-4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -----------------------------
# 3️⃣ TRANSFORMS
# -----------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet mean
        std=[0.229, 0.224, 0.225]    # ImageNet std
    )
])

# -----------------------------
# 4️⃣ DATASETS
# -----------------------------
train_dataset = datasets.ImageFolder(os.path.join(DATA_ROOT, "train"), transform=transform)
val_dataset = datasets.ImageFolder(os.path.join(DATA_ROOT, "val"), transform=transform)
test_seen_dataset = datasets.ImageFolder(os.path.join(DATA_ROOT, "test_seen"), transform=transform)
test_unseen_dataset = datasets.ImageFolder(os.path.join(DATA_ROOT, "test_unseen"), transform=transform)

# -----------------------------
# 5️⃣ DATALOADERS
# -----------------------------
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_seen_loader = DataLoader(test_seen_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_unseen_loader = DataLoader(test_unseen_dataset, batch_size=BATCH_SIZE, shuffle=False)

# -----------------------------
# 6️⃣ MODEL
# -----------------------------
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)  # Binary classification: real vs fake
model = model.to(DEVICE)

# -----------------------------
# 7️⃣ LOSS & OPTIMIZER
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# -----------------------------
# 8️⃣ EVALUATION FUNCTION (loss + accuracy)
# -----------------------------
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)

            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    accuracy = correct / total

    return avg_loss, accuracy

# -----------------------------
# 9️⃣ TRAINING LOOP
# -----------------------------
best_val_acc = 0.0
best_model_wts = copy.deepcopy(model.state_dict())

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    running_corrects = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        running_corrects += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = running_corrects / total

    val_loss, val_acc = evaluate(model, val_loader)

    print(f"\nEpoch [{epoch+1}/{NUM_EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

    # Save best model based on validation accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())

# -----------------------------
# 10️⃣ LOAD BEST MODEL
# -----------------------------
model.load_state_dict(best_model_wts)
torch.save(model.state_dict(), "/content/drive/MyDrive/best_resnet18_rgb_phase1.pth")
print("\nBest model saved to Drive.")

# -----------------------------
# 11️⃣ FINAL EVALUATION
# -----------------------------
train_loss, train_acc = evaluate(model, train_loader)
val_loss, val_acc = evaluate(model, val_loader)
test_seen_loss, test_seen_acc = evaluate(model, test_seen_loader)
test_unseen_loss, test_unseen_acc = evaluate(model, test_unseen_loader)

print("\n===== FINAL RESULTS =====")
print(f"Train Loss:       {train_loss:.4f} | Train Acc:       {train_acc:.4f}")
print(f"Validation Loss:  {val_loss:.4f} | Validation Acc:  {val_acc:.4f}")
print(f"Test Seen Loss:   {test_seen_loss:.4f} | Test Seen Acc:   {test_seen_acc:.4f}")
print(f"Test Unseen Loss: {test_unseen_loss:.4f} | Test Unseen Acc: {test_unseen_acc:.4f}")

Mounted at /content/drive
Using device: cuda


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 197MB/s]



Epoch [1/15]
Train Loss: 0.0152 | Train Acc: 0.9941
Val Loss:   0.0004 | Val Acc:   1.0000

Epoch [2/15]
Train Loss: 0.0010 | Train Acc: 1.0000
Val Loss:   0.0025 | Val Acc:   0.9990

Epoch [3/15]
Train Loss: 0.0016 | Train Acc: 0.9995
Val Loss:   0.0069 | Val Acc:   0.9990

Epoch [4/15]
Train Loss: 0.0031 | Train Acc: 0.9990
Val Loss:   0.0003 | Val Acc:   1.0000

Epoch [5/15]
Train Loss: 0.0009 | Train Acc: 0.9999
Val Loss:   0.0000 | Val Acc:   1.0000

Epoch [6/15]
Train Loss: 0.0001 | Train Acc: 1.0000
Val Loss:   0.0000 | Val Acc:   1.0000

Epoch [7/15]
Train Loss: 0.0001 | Train Acc: 1.0000
Val Loss:   0.0001 | Val Acc:   1.0000

Epoch [8/15]
Train Loss: 0.0001 | Train Acc: 1.0000
Val Loss:   0.0000 | Val Acc:   1.0000

Epoch [9/15]
Train Loss: 0.0000 | Train Acc: 1.0000
Val Loss:   0.0000 | Val Acc:   1.0000

Epoch [10/15]
Train Loss: 0.0000 | Train Acc: 1.0000
Val Loss:   0.0000 | Val Acc:   1.0000

Epoch [11/15]
Train Loss: 0.0000 | Train Acc: 1.0000
Val Loss:   0.0000 | Val 